# Optuna — モデル選択そのものを探索対象にする

**出典**　機械学習 4 分野の教科書 — 第 5 章 1-D 探索の自動化とモデル選定 ／ 06 Optuna：モデル選択そのものを探索対象にする

**このノートのパス**　`ml-code/05_hpo/06_optuna_model_selection.ipynb`

**このノートで確かめること**
**define-by-run** の性質を使い、「どのモデルを使うか」という分岐まで探索変数にする。
`storage` に履歴を残して**中断しても続きから再開できる**ことも確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `optuna xgboost lightgbm catboost scikit-learn` |
| 実行時間の目安 | 約 5〜10 分 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q optuna xgboost lightgbm catboost scikit-learn

In [3]:
import time, warnings
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import HyperbandPruner
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
SEED = 42

X, y = make_classification(
    n_samples=300_000, n_features=30, n_informative=15, n_redundant=8,
    weights=[0.95, 0.05], class_sep=1.3, random_state=SEED)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

# 探索用に Train をさらに fit / val に割る（Test には触らない）
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr[:12_000], y_tr[:12_000], test_size=0.25, stratify=y_tr[:12_000],
    random_state=SEED)
pos_weight = (y_fit == 0).sum() / (y_fit == 1).sum()
print("fit", X_fit.shape, "val", X_val.shape, "pos_weight", round(pos_weight, 2))

fit (9000, 30) val (3000, 30) pos_weight 17.4


## 目的関数 — モデル名を `suggest_categorical` で引き、分岐の中で別々の空間を作る

**パラメータ名にモデル名の接頭辞を付ける**のが肝。全モデルで `learning_rate` という同じ名前を使うと、Optuna は 1 つの軸として扱ってしまう。

In [4]:
def objective(trial):
    name = trial.suggest_categorical("model", ["xgboost", "lightgbm", "catboost"])

    if name == "xgboost":
        model = XGBClassifier(
            n_estimators = trial.suggest_int("xgb_n_estimators", 200, 500, step=100),
            max_depth    = trial.suggest_int("xgb_max_depth", 3, 12),
            learning_rate= trial.suggest_float("xgb_learning_rate", 1e-2, 3e-1, log=True),
            subsample    = trial.suggest_float("xgb_subsample", 0.6, 1.0),
            scale_pos_weight = pos_weight,        # ← 探索させない。データで決まる量
            random_state=SEED, n_jobs=-1, eval_metric="aucpr",
        )
    elif name == "lightgbm":
        model = LGBMClassifier(
            n_estimators = trial.suggest_int("lgb_n_estimators", 200, 500, step=100),
            num_leaves   = trial.suggest_int("lgb_num_leaves", 15, 120),
            learning_rate= trial.suggest_float("lgb_learning_rate", 1e-2, 3e-1, log=True),
            class_weight="balanced", random_state=SEED, n_jobs=-1, verbosity=-1,
        )
    else:
        model = CatBoostClassifier(
            iterations   = trial.suggest_int("cb_iterations", 200, 500, step=100),
            depth        = trial.suggest_int("cb_depth", 3, 10),
            learning_rate= trial.suggest_float("cb_learning_rate", 1e-2, 3e-1, log=True),
            l2_leaf_reg  = trial.suggest_float("cb_l2_leaf_reg", 1e-3, 10, log=True),
            scale_pos_weight=pos_weight, random_seed=SEED, verbose=0,
        )

    t0 = time.perf_counter()
    model.fit(X_fit, y_fit)
    trial.set_user_attr("fit_sec", time.perf_counter() - t0)
    trial.set_user_attr("model", name)

    prob = model.predict_proba(X_val)[:, 1]
    return average_precision_score(y_val, prob)     # 閾値に依存しない指標を返す

In [5]:
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),          # 既定。seed を渡すと再現できる
    pruner=HyperbandPruner(),               # TPE と組むなら公式の推奨はこちら
    study_name="model_selection",
    storage="sqlite:///study.db",           # 履歴をファイルに残す
    load_if_exists=True,                    # 中断しても続きから再開できる
)
study.optimize(objective, n_trials=12)      # 書籍では 200

print(round(study.best_value, 5))
print(study.best_params)

0.91326
{'model': 'catboost', 'cb_iterations': 500, 'cb_depth': 9, 'cb_learning_rate': 0.08881468338942386, 'cb_l2_leaf_reg': 3.3312804490899093}


## 全試行を DataFrame で取り出し、モデル別に見る

In [6]:
df = study.trials_dataframe()               # 全試行を DataFrame で取り出せる
print(df.shape, "件")

g = df.groupby("user_attrs_model").agg(
    試行数=("value", "size"), 最良=("value", "max"),
    中央値=("value", "median"), 平均学習秒=("user_attrs_fit_sec", "mean"))
print(g.round(4).to_string())

(48, 21) 件
                  試行数      最良     中央値   平均学習秒
user_attrs_model                             
catboost           34  0.9133  0.9058  1.5613
lightgbm            8  0.9034  0.9006  3.5799
xgboost             6  0.8898  0.8692  0.9637


### ハイパーパラメータ重要度（Optuna v5.0 の PED-ANOVA 拡張が効く場面）

モデルごとに違うパラメータを `suggest_*` する空間は**動的な探索空間**である。
Optuna v5.0 では、動的な空間でも重要度を計算できる拡張（KDD 2026 採択）が実装されている。

In [7]:
try:
    imp = optuna.importance.get_param_importances(study)
    for k, v in imp.items():
        print(f"{k:>22}: {v:.4f}")
except Exception as e:
    print("重要度の計算に失敗:", type(e).__name__, e)

        cb_l2_leaf_reg: 0.5997
      cb_learning_rate: 0.3172
         cb_iterations: 0.0645
              cb_depth: 0.0176
                 model: 0.0010
     xgb_learning_rate: 0.0000
         xgb_subsample: 0.0000
      xgb_n_estimators: 0.0000
         xgb_max_depth: 0.0000
      lgb_n_estimators: 0.0000
        lgb_num_leaves: 0.0000
     lgb_learning_rate: 0.0000


## 読み取りどころ

- **パラメータ名は必ずモデルごとに分ける**（`xgb_learning_rate` / `lgb_learning_rate`）。同名にすると別モデルの経験が混ざって学習される。
- **`scale_pos_weight` は探索させない。**データの陽性率から決まる量であって、探索して良くなる類のものではない。
- **モデル選択を入れると試行数を増やす必要がある。**3 モデルに分かれるので、実質 1 モデルあたりの試行数は 1/3 になる。
- **`storage="sqlite:///study.db"` + `load_if_exists=True` で中断・再開ができる。**同じ storage と study_name を指すプロセスを複数立てれば、それが**プロセス並列**にもなる。

⚠️ `study.db` がカレントディレクトリに作られる。やり直すときは削除するか `study_name` を変えること（さもないと前回の履歴に積み増す）。